# 7.2 · MCP Apps: interactive UI in the conversation

**MCP Apps：在对话里渲染交互界面**

> **After this lesson you can:** Build an MCP App with the Python SDK, linking a tool to a sandboxed `ui://` HTML resource with CSP metadata, and explain the host↔app postMessage protocol and its security model.

Official reading: [MCP Apps overview](https://modelcontextprotocol.io/extensions/apps/overview) · [Build an MCP App](https://modelcontextprotocol.io/extensions/apps/build)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. An MCP App server with the SDK's `Apps` extension
`mcp.server.apps.Apps` is an `Extension`. `@apps.tool(resource_uri=...)` stamps `_meta.ui` on a tool, and `add_html_resource(...)` serves a `ui://` resource as `text/html;profile=mcp-app` with optional CSP and permissions. The HTML below is a vanilla-JS app that speaks the postMessage protocol directly, with no SDK. Note the text fallback in `content`.

In [ ]:
import json

from pydantic import BaseModel

from mcp import Client
from mcp.client.extension import advertise
from mcp.server import MCPServer
from mcp.server.apps import (APP_MIME_TYPE, EXTENSION_ID, Apps, ResourceCsp,
                             client_supports_apps)
from mcp.server.mcpserver import Context

SALES = {"Q3": {"EMEA": 120, "AMER": 200, "APAC": 90}}
DASHBOARD_URI = "ui://sales/dashboard.html"

DASHBOARD_HTML = """<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>Sales by region</title></head>
<body>
  <h1 id="title">Loading…</h1>
  <ul id="bars"></ul>
  <button id="refresh">Refresh</button>
  <script>
    let nextId = 1; const pending = {};
    const send = (method, params) => new Promise(resolve => {
      const id = nextId++; pending[id] = resolve;
      window.parent.postMessage({jsonrpc: "2.0", id, method, params}, "*");
    });
    const notify = (method, params) => window.parent.postMessage({jsonrpc: "2.0", method, params}, "*");
    const render = (sc) => {
      document.getElementById("title").textContent = "Sales " + sc.quarter;
      document.getElementById("bars").innerHTML = Object.entries(sc.regions)
        .map(([r, v]) => `<li>${r}: ${"#".repeat(v / 10)} ${v}</li>`).join("");
    };
    window.addEventListener("message", (e) => {
      const m = e.data;
      if (m.id && pending[m.id]) { pending[m.id](m.result); delete pending[m.id]; }
      if (m.method === "ui/notifications/tool-result") render(m.params.structuredContent);
    });
    send("ui/initialize", {appInfo: {name: "sales-dashboard", version: "1.0.0"},
                           appCapabilities: {}, protocolVersion: "2026-01-26"})
      .then(() => notify("ui/notifications/initialized", {}));
    document.getElementById("refresh").onclick = async () => {
      const r = await send("tools/call", {name: "refresh_sales", arguments: {quarter: "Q3"}});
      render(r.structuredContent);
      send("ui/update-model-context", {content: [{type: "text", text: "User refreshed the Q3 dashboard."}]});
    };
  </script>
</body>
</html>"""


class RegionSales(BaseModel):
    quarter: str
    regions: dict[str, int]


apps = Apps()


@apps.tool(resource_uri=DASHBOARD_URI, description="Show sales by region for a quarter (interactive).")
def sales_by_region(quarter: str, ctx: Context) -> RegionSales:
    data = RegionSales(quarter=quarter, regions=SALES[quarter])
    return data   # the SDK puts this in structuredContent AND serializes it as text content


@apps.tool(resource_uri=DASHBOARD_URI, visibility=["app"], description="Re-fetch the dashboard data.")
def refresh_sales(quarter: str) -> RegionSales:
    return RegionSales(quarter=quarter, regions=SALES[quarter])


@apps.tool(resource_uri=DASHBOARD_URI, visibility=["model"],
           description="Email the quarter's report to someone (model only: the UI must never trigger emails).")
def email_report(quarter: str, to: str) -> str:
    return f"Sent the {quarter} report to {to}."


apps.add_html_resource(DASHBOARD_URI, DASHBOARD_HTML, name="Sales dashboard",
                       description="Bar chart of sales per region",
                       csp=ResourceCsp(connect_domains=["https://api.example.com"]),
                       prefers_border=True)

server = MCPServer("sales", extensions=[apps])   # extensions are applied here: register app tools first


@server.tool()
def export_csv(quarter: str, ctx: Context) -> str:
    """Export a quarter's sales as CSV text (model-facing, no UI)."""
    note = "" if client_supports_apps(ctx) else "  # tip: an Apps-capable host would show a dashboard"
    return "region,sales\n" + "\n".join(f"{r},{v}" for r, v in SALES[quarter].items()) + note


print("extension id:", EXTENSION_ID, "| mime type:", APP_MIME_TYPE)

### 2. Discovery: what an Apps-capable host sees
The host declares the extension with the **required** `mimeTypes` setting. It reads `_meta.ui` from `tools/list`, then fetches the template with `resources/read`.

In [ ]:
HOST_APPS_CAP = advertise(EXTENSION_ID, {"mimeTypes": [APP_MIME_TYPE]})

async with Client(server, extensions=[HOST_APPS_CAP]) as host:
    print("server/discover extensions:", host.server_capabilities.extensions)
    tools = (await host.list_tools()).tools
    for t in tools:
        print(f"tool {t.name:<16} _meta={json.dumps(t.meta)}")
    res = await host.read_resource(DASHBOARD_URI)
    item = res.contents[0].model_dump(by_alias=True, exclude_none=True)
    print("\nresources/read ->", json.dumps({k: v for k, v in item.items() if k != "text"}, indent=2))
    print("HTML bytes:", len(item["text"]))

### 3. Host-side validation: linkage, MIME type and the CSP header
A careful host checks every link before rendering anything. It confirms the `ui://` scheme, that the resource exists, the exact MIME type and the HTML5 doctype. It then builds the iframe's CSP **only** from declared domains, following the spec's construction, and writes the sandboxed document to a temp file with the policy injected as a `<meta>` tag.

In [ ]:
import tempfile
from html.parser import HTMLParser
from pathlib import Path


def build_csp(ui_meta: dict | None) -> str:
    csp = (ui_meta or {}).get("csp")
    if csp is None:   # spec's restrictive default when ui.csp is omitted
        return ("default-src 'none'; script-src 'self' 'unsafe-inline'; style-src 'self' 'unsafe-inline'; "
                "img-src 'self' data:; media-src 'self' data:; connect-src 'none';")
    res = csp.get("resourceDomains", [])
    directives = [
        ["default-src", "'none'"],
        ["script-src", "'self'", "'unsafe-inline'", *res],
        ["style-src", "'self'", "'unsafe-inline'", *res],
        ["connect-src", "'self'", *csp.get("connectDomains", [])],
        ["img-src", "'self'", "data:", *res],
        ["font-src", "'self'", *res],
        ["media-src", "'self'", "data:", *res],
        ["frame-src", *(csp.get("frameDomains") or ["'none'"])],
        ["object-src", "'none'"],
        ["base-uri", *(csp.get("baseUriDomains") or ["'self'"])],
    ]
    return "; ".join(" ".join(d) for d in directives) + ";"


class ExternalRefs(HTMLParser):
    def __init__(self):
        super().__init__()
        self.refs = []

    def handle_starttag(self, tag, attrs):
        for name, value in attrs:
            if name in ("src", "href") and value and value.startswith(("http://", "https://")):
                self.refs.append((tag, value))


async def validate_ui_links(client) -> dict:
    report = {}
    for tool in (await client.list_tools()).tools:
        uri = ((tool.meta or {}).get("ui") or {}).get("resourceUri")
        if not uri:
            continue
        assert uri.startswith("ui://"), f"{tool.name}: resourceUri must use ui://"
        content = (await client.read_resource(uri)).contents[0]
        assert content.mime_type == APP_MIME_TYPE, f"{uri}: wrong mimeType {content.mime_type}"
        assert content.text.lstrip().lower().startswith("<!doctype html>"), f"{uri}: not an HTML5 document"
        parser = ExternalRefs()
        parser.feed(content.text)
        report[tool.name] = {"uri": uri, "ui_meta": (content.meta or {}).get("ui"),
                             "external_refs": parser.refs, "html": content.text}
    return report


async with Client(server, extensions=[HOST_APPS_CAP]) as host:
    report = await validate_ui_links(host)

entry = report["sales_by_region"]
csp_header = build_csp(entry["ui_meta"])
print("linked tools:", {name: r["uri"] for name, r in report.items()})
print("external refs in HTML (must be covered by resourceDomains):", entry["external_refs"])
print("\nCSP for the iframe:\n ", csp_header.replace("; ", ";\n  "))

sandbox_dir = Path(tempfile.mkdtemp())
page = entry["html"].replace("<head>", f'<head><meta http-equiv="Content-Security-Policy" content="{csp_header}">', 1)
(sandbox_dir / "dashboard.html").write_text(page)
print("\nwrote", sandbox_dir / "dashboard.html", f"({len(page)} bytes) - open it in a browser to see the static shell")
print("default CSP if ui.csp were omitted:", build_csp(None))

### 4. Visibility: the model's tool list vs the app's
The host, not the server, enforces visibility. The agent sees only tools whose visibility includes `"model"`, which is the default when `visibility` is absent. The app may call only tools whose visibility includes `"app"`, also the default when absent. `email_report` is UI-linked but `["model"]`, so the model may send emails and the dashboard may not.

In [ ]:
def visibility(tool) -> list[str]:
    ui = (tool.meta or {}).get("ui") or {}
    return ui.get("visibility", ["model", "app"])   # spec default; a stricter host may refuse non-UI tools


async with Client(server, extensions=[HOST_APPS_CAP]) as host:
    all_tools = (await host.list_tools()).tools

model_tools = [t.name for t in all_tools if "model" in visibility(t)]
app_tools = [t.name for t in all_tools if "app" in visibility(t)]
print("tools the LLM is offered:   ", model_tools)
print("tools the app may call:     ", app_tools)
print("hidden from the LLM:        ", sorted(set(t.name for t in all_tools) - set(model_tools)))

### 5. Simulating the host ↔ app postMessage flow
Python can't run the iframe, but the protocol is plain JSON-RPC, so we play both sides. `FakeView` behaves like the HTML app above. `Host` renders it, completes the handshake, pushes tool input and result, and **proxies** the app's `tools/call` to the real MCP server after checking visibility. Every message is printed with its direction.

In [ ]:
class FakeView:
    """Stands in for the iframe: same messages the JS app sends."""
    def __init__(self):
        self.next_id, self.state = 1, None

    def request(self, method, params):
        msg = {"jsonrpc": "2.0", "id": self.next_id, "method": method, "params": params}
        self.next_id += 1
        return msg

    def on_message(self, msg):
        if msg.get("method") == "ui/notifications/tool-result":
            self.state = msg["params"]["structuredContent"]
            print(f"      [view renders] {self.state}")


class Host:
    def __init__(self, client, tools):
        self.client, self.tools = client, {t.name: t for t in tools}
        self.initialized, self.model_context = False, None

    def to_view(self, view, msg):
        assert self.initialized or msg.get("id"), "Host MUST NOT message the view before initialized"
        print("HOST -> APP ", json.dumps(msg)[:150])
        view.on_message(msg)

    async def from_view(self, view, msg):
        print("APP  -> HOST", json.dumps(msg)[:150])
        method, params = msg["method"], msg.get("params", {})
        if method == "ui/initialize":
            reply = {"protocolVersion": "2026-01-26", "hostInfo": {"name": "notebook-host", "version": "0.1"},
                     "hostCapabilities": {"serverTools": {}, "openLinks": {}},
                     "hostContext": {"theme": "dark", "displayMode": "inline",
                                     "containerDimensions": {"width": 400, "maxHeight": 600}}}
        elif method == "ui/notifications/initialized":
            self.initialized = True
            return
        elif method == "tools/call":
            tool = self.tools.get(params["name"])
            if tool is None or "app" not in visibility(tool):
                return self.to_view(view, {"jsonrpc": "2.0", "id": msg["id"],
                                           "error": {"code": -32000, "message": "Tool not callable from app"}})
            result = await self.client.call_tool(params["name"], params.get("arguments"))
            reply = result.model_dump(by_alias=True, exclude_none=True, mode="json")
        elif method == "ui/update-model-context":
            self.model_context, reply = params, {}
        else:
            reply = {}
        self.to_view(view, {"jsonrpc": "2.0", "id": msg["id"], "result": reply})


async with Client(server, extensions=[HOST_APPS_CAP]) as client:
    host, view = Host(client, (await client.list_tools()).tools), FakeView()
    args = {"quarter": "Q3"}                                   # the model decided to call sales_by_region
    await host.from_view(view, view.request("ui/initialize", {
        "appInfo": {"name": "sales-dashboard", "version": "1.0.0"},
        "appCapabilities": {}, "protocolVersion": "2026-01-26"}))
    await host.from_view(view, {"jsonrpc": "2.0", "method": "ui/notifications/initialized", "params": {}})
    host.to_view(view, {"jsonrpc": "2.0", "method": "ui/notifications/tool-input", "params": {"arguments": args}})
    result = await client.call_tool("sales_by_region", args)
    host.to_view(view, {"jsonrpc": "2.0", "method": "ui/notifications/tool-result",
                        "params": result.model_dump(by_alias=True, exclude_none=True, mode="json")})
    print("\n-- user clicks Refresh --")
    await host.from_view(view, view.request("tools/call", {"name": "refresh_sales", "arguments": args}))
    await host.from_view(view, view.request("ui/update-model-context", {
        "content": [{"type": "text", "text": "User refreshed the Q3 dashboard."}]}))
    print("\n-- a compromised app tries a model-only tool --")
    await host.from_view(view, view.request("tools/call", {"name": "email_report",
                                                               "arguments": {**args, "to": "attacker@example.com"}}))
    print("\nmodel context stored by host:", host.model_context)

### 6. Graceful degradation and guard rails
A host without Apps support still gets useful text: `content` always carries it, and the server can check `client_supports_apps`. The SDK also catches two common misconfigurations at startup: a tool bound to a resource that doesn't exist, and a URI without the `ui://` scheme.

In [ ]:
async with Client(server) as plain_host:     # no io.modelcontextprotocol/ui declared
    r = await plain_host.call_tool("sales_by_region", {"quarter": "Q3"})
    print("text a non-Apps host shows the model:", r.content[0].text.replace("\n", " "))
    print("export_csv in a non-Apps host:\n", (await plain_host.call_tool("export_csv", {"quarter": "Q3"})).content[0].text)

broken = Apps()


@broken.tool(resource_uri="ui://missing/app.html")
def orphan() -> str:
    return "never rendered"


try:
    MCPServer("broken", extensions=[broken])
except ValueError as exc:
    print("\nSDK refused the server:", exc)
try:
    broken.add_html_resource("https://evil.example/app.html", "<!DOCTYPE html>")
except ValueError as exc:
    print("SDK refused the resource:", exc)

### Your turn
1. Add `resource_domains=["https://cdn.jsdelivr.net"]` to the dashboard's `ResourceCsp` and a `<script src="https://cdn.jsdelivr.net/npm/chart.js">` tag to the HTML. Re-run section 3 and check that the external reference is now covered by `script-src`. Then remove the CSP entry and write a check that flags the uncovered reference.
2. Extend `Host.from_view` to handle `ui/message` by appending the message to a `chat` list, and `ui/open-link` by refusing any URL that is not `https`.
3. Make `sales_by_region` return a shorter text when `client_supports_apps(ctx)` is true, because the UI shows the details. What must the text still contain for the model to reason about the result?
4. Simulate teardown: have the host send the `ui/resource-teardown` **request** and wait for the view's response before "destroying" it.